# Caption + chữ trên màn hình cho toàn bộ keyframe BTC — Qwen2.5-VL qua vLLM

`config/ingest_shards_batch1_btc_caption.yaml`
Phương pháp: `notebooks/reference/caption.ipynb` (Bảo)

Chạy **sau** `02-keyframes-btc-import.ipynb`. Cùng bundle `aic_btc_b1_v1`, cùng volume
`keyframes-btc`, cùng bảng phân công — đổi `MEMBER` là ra đúng phần việc bạn đã nhập ở lượt trước.

### Hai tầng, một lần sinh

| Tầng | Artifact | Lấy từ |
|---|---|---|
| `n6_caption_qwen` | `qwen_captions` → `qwen_captions.json` | nửa `[CAPTION]` |
| `n6_ocr_qwen25` | `qwen_ocr` → `qwen_ocr.json` | nửa `[TEXT]` |

Cả hai đọc **cùng một** câu trả lời. `stages.qwen25_scan` giữ câu trả lời thô lại trong phiên
cho **một** video, nên tầng thứ hai không phải sinh lại — đó là một nửa tiền GPU của cả lượt.
Vì thế vòng lặp đi theo **VIDEO**, không theo tầng (giống notebook 06, khác notebook 03).

### Vì sao KHÔNG dùng lại artifact `dense_captions`

`dense_captions` là của `n6_recap`, và `spec.stage` là khoá **duy nhất** `merge.wanted` lọc theo.
Cướp nó đi là `n6_recap` thành một tầng không sinh ra gì — trong khi nó vẫn nằm trong
`config/ingest_shards.yaml`, kế hoạch 5 người của bundle `aic_btc_v1`.

Hai nguồn caption song song không phải sự trùng lặp cần dọn: ba nguồn OCR cũng đang sống cạnh
nhau vì chúng sai ở ba chỗ khác nhau.

---

## ⚠️ BA THỨ PHẢI ĐỌC TRƯỚC KHI BẤM RUN ALL

**1. `PART` chia việc, KHÔNG phải `MEMBER`.**

Muốn chạy nhiều notebook song song thì đổi `PART`, giữ nguyên `MEMBER`. Notebook cắt phần việc
của chính `MEMBER` đó thành `NUM_PARTS` lát rời nhau **theo gói zip** — nên mỗi lát chỉ tải zip
của nó, và `author` ghi ra vẫn là `MEMBER`.

Thêm một tên mới vào `assignment` để chạy song song là **cách chắc chắn làm hỏng bundle**:
`part-manifest/{video}.json` ghi đúng **một** `author`, và `merge.check_ownership` từ chối cả
bundle nếu author khác người được giao (§4.1 handoff).

Hai lát không bao giờ chạm cùng một video, nên `write_part_manifest(merge=True)` — vốn là
read-modify-write — vẫn an toàn khi chạy song song. Chia lát theo cách khác thì mất tính chất đó.

*Giới hạn đã đo:* chia theo gói nên **một gói không cắt nhỏ được**. `Keyframes_L25.zip` có
37.445 ảnh trên tổng 88.664 của `trong`, nên với `trong` phép chia chỉ cân tới `NUM_PARTS = 2`
(44.226 / 44.438); từ 3 lát trở lên lát chứa L25 luôn là lát nặng nhất và các lát kia chờ nó.
`quang` chia đều tới 4 lát (16.7k–25.6k). Ô Step 0b **in bảng tải của mọi lát** trước khi chạy —
đọc nó rồi mới chọn `NUM_PARTS`.

**2. Notebook này KHÔNG được chạy chung phiên với 03 / 06.**

vLLM ghim `torch` riêng và sẽ kéo phiên bản khác về khi cài. Đổi torch dưới chân notebook 03 là
đổi tiền xử lý ảnh của SigLIP, tức vector của bundle thành hỗn hợp hai phép đo — và
`merge_bundle.py` sẽ từ chối cả bundle ở bước gộp, sau nhiều giờ. Một phiên, một môi trường.

**3. `PROBE_ONLY = True` ở lần chạy đầu.**

Nó chạy `L21_V001` rồi **đối chiếu với bản Bảo đã chạy** và dừng. Bảo để lại 855 keyframe của
`L21_V001..V003` ở `notebooks/reference/dense_captions_qwen.json` — đó là bản đối chiếu **duy
nhất** ta có. Lượt đầy đủ là 177.321 ảnh trên một model 7B; phát hiện lệch *sau* lượt đó là mất
cả lượt.

## Dùng lại private Kaggle cache

Attach cùng Dataset đã dùng cho notebook 02/03. Kaggle thường tự giải nén archive thành thư mục;
notebook nhận cả layout đã giải nén lẫn ZIP nguyên bản trong `/kaggle/input`, giữ cache read-only
và chỉ tải từ website khi cả hai dạng đều thiếu. Không copy 15 GiB sang `/kaggle/working`.

## Step 0a — repo + gói

In [ ]:
MEMBER      = "trong"          # <-- ĐỔI THÀNH TÊN BẠN. KHÔNG dùng để chia song song.
PART        = 0                # <-- lát thứ mấy (0-based). ĐÂY là cần gạt chia song song.
NUM_PARTS   = 1                # <-- tổng số notebook đang chạy cho CÙNG `MEMBER` này.

SHARDS      = "config/ingest_shards_batch1_btc_caption.yaml"
INVENTORY   = "config/zips_keyframes_btc.yaml"
TRANSPORT   = "modal"
VOLUME      = "keyframes-btc"  # CÙNG volume với notebook 02/03/04/06
VOL_VERSION = 2

# ── ĐỐI CHIẾU TRƯỚC KHI SCALE ─────────────────────────────────────────────
# True  = chạy PROBE_VIDEOS, so với bản của Bảo, rồi DỪNG. Không ghi gì lên Volume.
# False = chạy thật phần việc của lát này.
#
# Để True ở lần chạy đầu tiên của MỖI người. Nó tốn ~1 gói zip + vài phút GPU, và
# nó là thứ duy nhất đứng giữa một lỗi địa chỉ và 177.321 ảnh chạy sai.
PROBE_ONLY   = True
PROBE_VIDEOS = ["L21_V001"]    # 307 ảnh, nằm trong bản đối chiếu của Bảo

# ── PHẦN CỨNG ─────────────────────────────────────────────────────────────
# 2 = Kaggle "T4 x2" (mặc định của Bảo): cưa đôi model 7B ra hai card.
# 1 = Kaggle P100 hoặc T4 đơn — PHẢI hạ MAX_MODEL_LEN xuống ~2048 và BATCH xuống ~8.
#
# Hai giá trị này đi vào `facts` của marker chứ KHÔNG vào `params`: chúng là lựa
# chọn của MÁY. Đặt vào `params` thì `check_fingerprints` từ chối cả bundle chỉ vì
# hai người dùng hai loại GPU — một phép từ chối đúng hình thức và rỗng về nội dung.
TP_SIZE       = 2
BATCH         = 32
MAX_MODEL_LEN = 4096

# ── CHẠY THỬ / CHẠY LẠI ───────────────────────────────────────────────────
ONLY_VIDEOS = []   # [] = cả lát của bạn
REDO_STAGES = []   # chạy lại DÙ marker đã báo xong (ghi đè part + marker cũ)

REPO_URL    = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"
# ⚠️ MỌI NGƯỜI CHẠY BUNDLE NÀY PHẢI ĐỂ CÙNG MỘT NHÁNH — xem notebook 03.
REPO_REF    = "feature/triet/structure"

import os, subprocess, sys
from pathlib import Path

REPO = Path("/kaggle/input/mfe-repo")
if not REPO.is_dir():
    REPO = Path("/kaggle/working/mfe")
    if not REPO.is_dir():
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        url = REPO_URL.replace("https://", f"https://{token}@")
        subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_REF, url, str(REPO)],
                       check=True, capture_output=True)   # token không lọt vào log
sys.path.insert(0, str(REPO / "src"))

req = REPO / "notebooks" / "requirements-kaggle-btc-caption.txt"
print(f"cài từ {req.name}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)

# PHIÊN BẢN vLLM PHẢI GIỐNG NHAU GIỮA HAI NGƯỜI.
#
# `vllm` nằm trong `NUMERIC_PACKAGES`: đổi bản là đổi cách xếp lịch lô và cách
# lượng tử hoá KV cache, tức đổi chuỗi model trả về. `merge_bundle.py` TỪ CHỐI
# bundle có hai bản khác nhau — và nó chỉ nói ra ở bước gộp, sau nhiều giờ.
# Chép con số dưới đây cho người chạy lát còn lại.
from importlib.metadata import version
print(f"\n>>> vllm=={version('vllm')}  — CHÉP CON SỐ NÀY cho người chạy lát khác <<<")

## Step 0b — Modal, phân công, cắt lát

In [ ]:
import logging, yaml
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

from kaggle_secrets import UserSecretsClient
from mfe.offline.bundle import BundleLayout, BundleWriter, StageStatus, load_shard_plan
from mfe.offline.stages import STAGES, StageContext, read_keyframes, unload_models
from mfe.offline.transport import build_transport

secrets = UserSecretsClient()
os.environ["MODAL_TOKEN_ID"] = secrets.get_secret("MODAL_TOKEN_ID")
os.environ["MODAL_TOKEN_SECRET"] = secrets.get_secret("MODAL_TOKEN_SECRET")

RAW       = yaml.safe_load((REPO / SHARDS).read_text(encoding="utf-8"))
INV       = yaml.safe_load((REPO / INVENTORY).read_text(encoding="utf-8"))
plan      = load_shard_plan(REPO / SHARDS)
layout    = BundleLayout(plan.bundle)
transport = build_transport(TRANSPORT, volume=VOLUME,
                            local_root="/kaggle/working/bundle_local",
                            version=VOL_VERSION)
WORK   = Path("/kaggle/working/scratch"); WORK.mkdir(parents=True, exist_ok=True)
ZIPS   = Path("/kaggle/working/zips");    ZIPS.mkdir(parents=True, exist_ok=True)
KAGGLE_INPUT = Path("/kaggle/input")
writer = BundleWriter(transport, layout, MEMBER, WORK)

assert plan.keyframe_set == "btc"
STAGE_LIST = list(plan.stages)          # ['n6_caption_qwen', 'n6_ocr_qwen25'] — THỨ TỰ quan trọng
PARAMS     = RAW.get("params", {})
EXTRAS     = RAW["extras"]

# Tham số của MÁY, chèn vào params lúc chạy (xem ô Step 0a về vì sao không nằm trong yaml).
MACHINE = {"tensor_parallel_size": TP_SIZE, "batch_size": BATCH, "max_model_len": MAX_MODEL_LEN}


def split_zips(zips, num_parts, part):
    """Chia gói zip cho `num_parts` lát, cân theo SỐ ẢNH. Trả về phần của `part`.

    Chia theo GÓI chứ không theo video, và đó là chuyện chi phí chứ không phải
    thẩm mỹ: vòng lặp tải cả gói zip rồi mới bung từng video. Chia xen kẽ theo
    video thì mọi lát đều phải tải mọi gói — 7 lát là 7 lần tải 28,7 GB.

    Xếp gói to trước rồi thả vào lát đang nhẹ nhất (LPT). Tất định, nên hai lát
    chạy trên hai máy khác nhau vẫn ra đúng cùng một phép chia.
    """
    sized = sorted(
        zips, key=lambda z: -sum(INV["zips"][z[0]]["counts"].values())
    )
    buckets = [[] for _ in range(num_parts)]
    loads = [0] * num_parts
    for entry in sized:
        target = loads.index(min(loads))
        buckets[target].append(entry)
        loads[target] += sum(INV["zips"][entry[0]]["counts"].values())
    for index, (bucket, load) in enumerate(zip(buckets, loads)):
        mark = " <-- LÁT NÀY" if index == part else ""
        print(f"  lát {index}: {load:>6} ảnh  {', '.join(n for n, _ in bucket)}{mark}")
    return buckets[part]


if not 0 <= PART < NUM_PARTS:
    raise SystemExit(f"PART={PART} phải nằm trong 0..{NUM_PARTS - 1}")

ALL_MY_ZIPS = plan.zips_for(MEMBER)
print(f"phân công của {MEMBER}: {len(ALL_MY_ZIPS)} gói, cắt thành {NUM_PARTS} lát")
MY_ZIPS   = split_zips(ALL_MY_ZIPS, NUM_PARTS, PART)
mine      = {v for name, _ in MY_ZIPS for v in plan.videos_in(name)}
MY_VIDEOS = [v for v in plan.videos_for(MEMBER) if v in mine]
status    = StageStatus.load(transport, layout, MEMBER)

# Điều kiện tiên quyết: notebook 02 phải xong TRƯỚC (map-keyframes trên Volume).
missing_import = [v for v in MY_VIDEOS if not status.is_done("n0_import_btc", v)]
todo = [(s, v) for v in MY_VIDEOS for s in STAGE_LIST if not status.is_done(s, v)]

print()
print(f"bundle  : {plan.bundle}  ->  volume {VOLUME} (v{VOL_VERSION})")
print(f"người   : {MEMBER}  ·  lát {PART}/{NUM_PARTS}")
print(f"video   : {len(MY_VIDEOS)} (cả phần của {MEMBER}: {len(plan.videos_for(MEMBER))})")
print(f"tầng    : {', '.join(STAGE_LIST)}")
print(f"còn lại : {len(todo)} việc / {len(MY_VIDEOS) * len(STAGE_LIST)}")
if missing_import:
    print(f"\nCHU Y: {len(missing_import)} video CHUA nhap bang notebook 02 — se BO QUA")
if PROBE_ONLY:
    print(f"\n>>> PROBE_ONLY: chi chay {PROBE_VIDEOS} de doi chieu, KHONG ghi len Volume <<<")

## Step 0c — tải gói, bung ảnh (giống notebook 03/06)

In [ ]:
import shutil, zipfile

def free_gb():
    return shutil.disk_usage("/kaggle/working").free / 2**30

def check_size(target, expect):
    if expect is None:
        return
    actual = target.stat().st_size
    if actual != expect:
        target.unlink(missing_ok=True)
        raise RuntimeError(
            f"{target.name}: tai ve {actual:,} byte, cho doi {expect:,} "
            f"(lech {actual - expect:+,}). Da xoa file hong — chay lai o nay."
        )

def validate_zip(path):
    try:
        with zipfile.ZipFile(path) as archive:
            if not archive.infolist():
                raise RuntimeError(f"{path}: ZIP rỗng")
    except zipfile.BadZipFile as err:
        raise RuntimeError(f"{path}: không phải ZIP hoàn chỉnh") from err

def find_kaggle_zip(name, expect_bytes=None):
    if not KAGGLE_INPUT.is_dir():
        return None
    candidates = sorted(p for p in KAGGLE_INPUT.rglob(name) if p.is_file())
    if not candidates:
        return None
    if len(candidates) != 1:
        found = ", ".join(str(p) for p in candidates)
        raise RuntimeError(f"{name}: thấy nhiều bản trong /kaggle/input: {found}")
    cached = candidates[0]
    actual = cached.stat().st_size
    if expect_bytes is not None and actual != expect_bytes:
        raise RuntimeError(f"{cached}: {actual:,} byte, chờ đợi {expect_bytes:,}")
    validate_zip(cached)
    print(f"    cache Kaggle: {cached} ({actual / 2**30:.2f} GB)")
    return cached

def download(url, name, expect_bytes=None):
    cached = find_kaggle_zip(name, expect_bytes)
    if cached is not None:
        return cached
    target = ZIPS / name
    if target.is_file():
        size = target.stat().st_size
        complete = expect_bytes is not None and size == expect_bytes
        if expect_bytes is None:
            try:
                validate_zip(target); complete = True
            except RuntimeError:
                complete = False
        if complete:
            print(f"    {name}: đã có ({size / 2**30:.2f} GB)")
            return target
        print(f"    {name}: đang dở, chạy tiếp")
    print(f"    tải {name} … (đĩa trống {free_gb():.1f} GB)")
    # `--read-timeout` la thu quan trong nhat: khong co no, mot ket noi bi bop
    # xuong ~400 B/s van la ket noi "song" va wget ngoi doi vo han.
    subprocess.run(
        ["wget", "-q", "--show-progress", "-c",
         "--timeout=30", "--read-timeout=60", "--tries=0", "--waitretry=10",
         "--retry-connrefused", "-O", str(target), url],
        check=True,
    )
    check_size(target, expect_bytes)
    try:
        validate_zip(target)
    except RuntimeError as err:
        target.unlink(missing_ok=True)
        raise RuntimeError(f"{name}: tải xong nhưng ZIP hỏng; đã xóa, chạy lại") from err
    return target

def index_kaggle_extracted_archives(names):
    targets = {Path(name).stem for name in names}
    found = {name: [] for name in targets}
    if not KAGGLE_INPUT.is_dir():
        return {}
    for root, directories, _files in os.walk(KAGGLE_INPUT):
        keep = []
        for directory in directories:
            if directory in targets:
                found[directory].append(Path(root) / directory)
            else:
                keep.append(directory)
        directories[:] = keep      # không đi sâu vào một archive đã khớp
    indexed = {}
    for name, paths in found.items():
        if len(paths) > 1:
            raise RuntimeError(f"{name}: nhiều cache đã giải nén: "
                               f"{', '.join(str(p) for p in paths)}")
        if paths:
            indexed[name] = paths[0]
    return indexed

PROBE_ZIPS = []
if PROBE_ONLY:
    # Doc tu `plan.members` chu khong go ten: video doi chieu (L21_*) thuoc phan
    # cua `trong`, nhung ai cung phai chay duoc cong doi chieu — lat thu KHONG
    # ghi gi len Volume nen quyen so huu khong lien quan o day.
    for member in plan.members:
        for name, url in plan.zips_for(member):
            if (name, url) in PROBE_ZIPS:
                continue
            if any(v in plan.videos_in(name) for v in PROBE_VIDEOS):
                PROBE_ZIPS.append((name, url))
    missing_probe = [
        v for v in PROBE_VIDEOS
        if not any(v in plan.videos_in(name) for name, _ in PROBE_ZIPS)
    ]
    if missing_probe:
        raise SystemExit(f"PROBE_VIDEOS khong co trong goi nao: {missing_probe}")

WANTED_ZIPS = PROBE_ZIPS if PROBE_ONLY else MY_ZIPS
archive_names = [name for name, _url in WANTED_ZIPS] + ["map-keyframes-aic25-b1.zip"]
EXTRACTED_ARCHIVES = index_kaggle_extracted_archives(archive_names)

def find_kaggle_extracted_dir(archive_name, child):
    archive = EXTRACTED_ARCHIVES.get(Path(archive_name).stem)
    if archive is None:
        return None
    extracted = archive / child
    if not extracted.is_dir():
        raise RuntimeError(f"{archive}: thiếu thư mục {child}")
    print(f"    cache Kaggle đã giải nén: {extracted}")
    return extracted

def extract_video(zip_path, video_id, into):
    """Bung ảnh của ĐÚNG một video. Bung cả gói L25 là 5,67 GB nằm chờ vô ích."""
    out = into / video_id
    out.mkdir(parents=True, exist_ok=True)
    count = 0
    with zipfile.ZipFile(zip_path) as archive:
        for name in archive.namelist():
            parts = Path(name).parts
            if len(parts) < 2 or parts[-2] != video_id or name.endswith("/"):
                continue
            with archive.open(name) as src, (out / parts[-1]).open("wb") as dst:
                shutil.copyfileobj(src, dst, length=1 << 20)
            count += 1
    return out, count

def extract_one_file(zip_path, inner_suffix, dest):
    with zipfile.ZipFile(zip_path) as archive:
        match = [n for n in archive.namelist() if n.endswith(inner_suffix)]
        if not match:
            raise FileNotFoundError(f"{inner_suffix} khong co trong {zip_path.name}")
        dest.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(match[0]) as src, dest.open("wb") as dst:
            shutil.copyfileobj(src, dst)
    return dest

print("gói dùng chung:")
MAPS_DIR = find_kaggle_extracted_dir("map-keyframes-aic25-b1.zip", "map-keyframes")
MAPS_ZIP = None if MAPS_DIR is not None else download(
    EXTRAS["map_keyframes"], "map-keyframes-aic25-b1.zip"
)

def prepare(video_id, zip_name, kf_root, kf_zip):
    """(thư mục ảnh, số ảnh, đường dẫn map csv) cho một video."""
    if kf_root is not None:
        kf_dir = kf_root / video_id
        if not kf_dir.is_dir():
            raise FileNotFoundError(f"{video_id}: thiếu trong {kf_root}")
        n_img = sum(1 for p in kf_dir.iterdir()
                    if p.is_file() and p.suffix.lower() == ".jpg")
    else:
        kf_dir, n_img = extract_video(kf_zip, video_id, WORK / "keyframes")
    if MAPS_DIR is not None:
        map_csv = MAPS_DIR / f"{video_id}.csv"
        if not map_csv.is_file():
            raise FileNotFoundError(f"{map_csv}: không tồn tại")
    else:
        map_csv = extract_one_file(MAPS_ZIP, f"/{video_id}.csv",
                                   WORK / "map-keyframes" / f"{video_id}.csv")
    return kf_dir, n_img, map_csv

print(f"đĩa trống: {free_gb():.1f} GB")

## Step 1 — ĐỐI CHIẾU với bản Bảo đã chạy

Chỉ chạy khi `PROBE_ONLY = True`. Nó sinh cho `PROBE_VIDEOS`, ghi ra một file **cục bộ** theo
đúng định dạng khoá của Bảo (`{video}_{n}`), rồi gọi `scripts/compare_qwen_captions.py`.

**Không ghi gì lên Volume.** Một lượt thử phải là một lượt thử.

Ba trục được so, và không trục nào là "hai chuỗi giống hệt nhau" — `temperature=0.1` trên hai GPU
khác nhau sẽ không bao giờ cho ra cùng từng ký tự:

| Trục | Lệch nghĩa là |
|---|---|
| **phủ khoá** | lỗi **địa chỉ** — `n` chưa được đổi sang `frame_idx` qua map-keyframes |
| **tỉ lệ có đủ `[CAPTION]`+`[TEXT]`** | prompt hỏng thầm lặng, hoặc `max_tokens` cắt mất `[TEXT]` |
| **chồng lấn từ vựng** | đang mô tả **hai ảnh khác nhau** — cũng là lỗi địa chỉ |

Không đạt thì ô này `SystemExit`, và đó là đúng: đừng scale.

In [ ]:
if PROBE_ONLY:
    import json as _json
    from mfe.offline.stages import parse_qwen25_caption_ocr, qwen25_scan

    probe_raw = {}
    for zip_name, zip_url in PROBE_ZIPS:
        kf_root = find_kaggle_extracted_dir(zip_name, "keyframes")
        kf_zip = None if kf_root is not None else download(
            zip_url, zip_name, plan.zip_bytes.get(zip_name)
        )
        for video_id in [v for v in PROBE_VIDEOS if v in plan.videos_in(zip_name)]:
            kf_dir, n_img, map_csv = prepare(video_id, zip_name, kf_root, kf_zip)
            keyframes = read_keyframes(map_csv, kf_dir)
            print(f"  {video_id}: {n_img} ảnh, {len(keyframes)} dòng map-keyframes")

            ctx = StageContext(video_id, Path("/dev/null"), keyframes, writer, layout, WORK)
            ctx.params = {**PARAMS[STAGE_LIST[0]], **MACHINE}
            answers = qwen25_scan(ctx, keyframes, label=f"probe {video_id}")
            # Khoa theo dinh dang cua Bao (`{video}_{n}`) de script doi chieu tu
            # quy no ve khoa cua kho — no lam viec do bang map-keyframes, tuc bang
            # dung cho duy nhat noi `n` voi `frame_idx`.
            for keyframe, answer in zip(keyframes, answers):
                probe_raw[f"{video_id}_{keyframe.n:03d}"] = answer

            marked = sum(
                1 for a in answers if parse_qwen25_caption_ocr(a)[2]
            )
            print(f"     có đủ hai nhãn: {marked}/{len(answers)}")
            shutil.rmtree(WORK / "keyframes" / video_id, ignore_errors=True)

    probe_path = Path("/kaggle/working/probe_qwen25.json")
    probe_path.write_text(_json.dumps(probe_raw, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"\nđã ghi {probe_path} ({len(probe_raw)} khoá)")

    maps_arg = str(MAPS_DIR) if MAPS_DIR is not None else str(WORK / "map-keyframes")
    if MAPS_DIR is None:
        # Script doi chieu can CA thu muc map-keyframes; bung nhung video da probe.
        for video_id in PROBE_VIDEOS:
            extract_one_file(MAPS_ZIP, f"/{video_id}.csv",
                             WORK / "map-keyframes" / f"{video_id}.csv")

    result = subprocess.run(
        [sys.executable, str(REPO / "scripts" / "compare_qwen_captions.py"),
         "--ours", str(probe_path),
         "--theirs", str(REPO / "notebooks" / "reference" / "dense_captions_qwen.json"),
         "--maps", maps_arg],
        cwd=str(REPO), capture_output=True, text=True,
    )
    print(result.stdout)
    if result.stderr.strip():
        print(result.stderr)
    if result.returncode != 0:
        raise SystemExit("ĐỐI CHIẾU KHÔNG ĐẠT — đọc bảng trên, đừng đặt PROBE_ONLY = False")
    print("\n>>> ĐẠT. Đặt PROBE_ONLY = False rồi Run All lại để chạy thật. <<<")
else:
    print("bỏ qua đối chiếu (PROBE_ONLY = False)")

## Step 2 — vòng lặp thật

Theo **video**, không theo tầng: hai tầng đọc hai nửa của cùng một câu trả lời, và
`qwen25_scan` chỉ giữ câu trả lời lại cho **một** video. Chạy theo tầng là sinh lại toàn bộ ở
tầng thứ hai — gấp đôi tiền GPU, không lỗi nào báo.

Marker cắm theo từng *(tầng, video)*. Phiên đứt giữa chừng thì lần chạy sau bỏ đúng những việc đã
xong, kể cả khi nó đứt giữa hai tầng của cùng một video.

`merge=True` khi ghi manifest: bundle này được đổ bởi **năm** notebook (02 nhập, 03 ảnh, 04 video,
06 OCR, 08 đây). Không gộp thì mỗi lượt xoá sạch artifact của lượt trước, còn marker vẫn báo
"xong" nên không lần chạy nào phục hồi lại.

In [ ]:
import time, traceback

def stages_for(video_id):
    return [s for s in STAGE_LIST
            if s in REDO_STAGES or not status.is_done(s, video_id)]

failures = []
started = time.perf_counter()
done_count = 0

def run_one(video_id, kf_dir, map_csv, stages):
    keyframes = read_keyframes(map_csv, kf_dir)
    ctx = StageContext(video_id, Path("/dev/null"), keyframes, writer, layout, WORK)
    markers = []
    for stage in stages:
        # Tham so cua MAY tron vao tham so cua BUNDLE. `_qwen25_common` tach lai
        # dung cho: cai nao vao `params` (so giua cac shard), cai nao vao `facts`.
        ctx.params = {**PARAMS.get(stage, {}), **MACHINE}
        marker = STAGES[stage](ctx)
        markers.append(marker)
        status.done.add((stage, video_id))
        print(f"      {stage:<18} {marker.get('rows', '-'):>6} dòng  "
              f"{marker.get('unmarked', '-'):>4} mất nhãn  {marker.get('elapsed_s')}s")
    return keyframes, markers

if PROBE_ONLY:
    print("PROBE_ONLY = True — bỏ qua vòng lặp thật.")
else:
    for zip_name, zip_url in MY_ZIPS:
        wanted = set(MY_VIDEOS) & set(ONLY_VIDEOS or MY_VIDEOS)
        videos = [v for v in plan.videos_in(zip_name)
                  if v in wanted and status.is_done("n0_import_btc", v)]
        pending = [v for v in videos if stages_for(v)]
        if not pending:
            print(f"\n=== {zip_name}: xong hết, không tải ===")
            continue

        print(f"\n=== {zip_name}: còn {len(pending)}/{len(videos)} video ===")
        kf_root = find_kaggle_extracted_dir(zip_name, "keyframes")
        kf_zip = None if kf_root is not None else download(
            zip_url, zip_name, plan.zip_bytes.get(zip_name)
        )
        zip_started, zip_done = time.perf_counter(), 0

        for video_id in pending:
            stages = stages_for(video_id)
            try:
                kf_dir, n_img, map_csv = prepare(video_id, zip_name, kf_root, kf_zip)
                print(f"  {video_id}  {n_img} ảnh  ({len(stages)} tầng)")
                keyframes, markers = run_one(video_id, kf_dir, map_csv, stages)
                writer.write_part_manifest(video_id, markers,
                                           keyframes=len(keyframes), merge=True)
                done_count += 1; zip_done += 1
                left = (len(pending) - zip_done) * (time.perf_counter() - zip_started) / zip_done
                print(f"  [{done_count:>3}] {video_id} xong | còn ~{left / 3600:.1f}h cho gói này")
            except Exception as err:
                failures.append((video_id, repr(err)))
                print(f"  [LỖI] {video_id}: {err}")
                traceback.print_exc(limit=3)
            finally:
                shutil.rmtree(WORK / "keyframes" / video_id, ignore_errors=True)
                (WORK / "map-keyframes" / f"{video_id}.csv").unlink(missing_ok=True)

        if kf_zip is not None and kf_zip.parent == ZIPS:
            kf_zip.unlink(missing_ok=True)
            print(f"    đã xoá {zip_name}, đĩa trống {free_gb():.1f} GB")

    print()
    print("=" * 72)
    print(f"phiên này: {done_count} video, {len(failures)} lỗi, "
          f"{(time.perf_counter() - started) / 3600:.1f}h")
    for video_id, err in failures:
        print(f"  {video_id}: {err}")

## Step 3 — chốt phiên

In [ ]:
unload_models()

fresh = StageStatus.load(transport, layout, MEMBER)
left  = [(s, v) for v in MY_VIDEOS for s in STAGE_LIST if not fresh.is_done(s, v)]
total = len(MY_VIDEOS) * len(STAGE_LIST)
print(f"{MEMBER} lát {PART}/{NUM_PARTS}: {total - len(left)}/{total} việc đã lên {VOLUME}")
for stage in STAGE_LIST:
    n = sum(1 for v in MY_VIDEOS if fresh.is_done(stage, v))
    print(f"  {stage:<18} {n:>4}/{len(MY_VIDEOS)} video")

# Tien do CA phan cua MEMBER, khong chi lat nay — de biet khi nao ca nguoi do xong.
all_mine = plan.videos_for(MEMBER)
whole = [(s, v) for v in all_mine for s in STAGE_LIST if not fresh.is_done(s, v)]
print(f"\ncả phần của {MEMBER}: {len(all_mine) * len(STAGE_LIST) - len(whole)}"
      f"/{len(all_mine) * len(STAGE_LIST)} việc")

if left:
    print(f"\ncòn {len(left)} việc trong lát này — bấm Run All lại, nó chạy tiếp từ chỗ đứt")
else:
    print("\nXONG lát này. Báo Lead.")
    print(f"  python scripts/verify_bundle.py --shards {SHARDS} --member {MEMBER}")